In [ ]:
import requests
import matplotlib.pyplot as plt
import pandas as pd
import json
import csv
import numpy as np

In [ ]:
%pip install google-analytics-data pandas

# Google Analytics API

In [ ]:
# PROPERTY ID & KEY FILE

from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.oauth2 import service_account

PROPERTY_ID = "321460044"

KEY_FILE = r"C:\Users\jlmow\Documents-C Drive\NSS-C Drive\Capstone\sfs-mrktg-76749b6efce7.json"

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE
)

client = BetaAnalyticsDataClient(credentials=credentials)

print("Connection setup completed.")

## API Pull: GA1 - Dates, Sessions, Source/Medium

In [ ]:
from datetime import date, timedelta

# First day of the current month
first_day_this_month = date.today().replace(day=1)

# Last day of the previous month
last_day_last_month = first_day_this_month - timedelta(days=1)

In [ ]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        ],
    metrics=[
        Metric(name="sessions")
    ],
    date_ranges=[
        DateRange(
            start_date="1470daysAgo",
            end_date=last_day_last_month.strftime("%Y-%m-%d")
        )
    ],
 limit=100000
)
response_ga1 = client.run_report(request)

print("Number of rows returned:", len(response_ga1.rows))

In [ ]:
response_ga1

In [ ]:
data = []

for row in response_ga1.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "sessions": row.metric_values[0].value
    })

ga1 = pd.DataFrame(data)

ga1["date"] = pd.to_datetime(
    ga1["date"],
    format="%Y%m%d"
)

ga1["sessions"] = pd.to_numeric(
    ga1["sessions"]

)

ga1.sort_values(by="date")

In [ ]:
ga1['month'] = ga1['date'].dt.month
ga1['year'] = ga1['date'].dt.year
ga1

In [ ]:
ga1.to_csv('ga1.csv', index=False)

### Sessions grouped by year, then month

In [ ]:
ga1_grouped = ga1[['year','month','sessions']].groupby(['year','month']).sum().reset_index()
ga1_grouped

### Save DF to .csv

In [ ]:
ga1_grouped.to_csv('ga1_grouped.csv', index=False)

## API Pull: GA2 - Date, Sessions, Source/Medium, Page Paths

In [ ]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        Dimension(name="pagePath")
    ],
    metrics=[
        Metric(name="sessions")
    ],
    date_ranges=[
        DateRange(
            start_date="761daysAgo",
            end_date="yesterday"
        )
    ],
 limit=100000
)
response_ga2 = client.run_report(request)

print("Number of rows returned:", len(response_ga2.rows))

In [ ]:
response_ga2

In [ ]:
data = []

for row in response_ga2.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "page_path": row.dimension_values[2].value,
        "sessions": row.metric_values[0].value
    })

ga2 = pd.DataFrame(data)

ga2["date"] = pd.to_datetime(
    ga2["date"],
    format="%Y%m%d"
)

ga2["sessions"] = pd.to_numeric(
    ga2["sessions"]

)

ga2.sort_values(by="date")

In [ ]:
ga2['sessions'].sum()

In [ ]:
ga2.dtypes

In [ ]:
ga2['month'] = ga2['date'].dt.month
ga2['year'] = ga2['date'].dt.year
ga2

### Group by Month/Year

In [ ]:
ga2_sessions = ga2[['year','month','sessions']].groupby(['year','month']).sum()
ga2_sessions

### GA2 - Sum by source_medium

In [ ]:
ga2_date = ga2.groupby(['source_medium','date']).sum()
ga2_date

### GA2 - Sum by Date

In [ ]:
ga2_all = ga2.groupby(['date']).sum()
ga2
#sum with text concatenates

# HubSpot API

In [ ]:
%pip install requests pandas

In [ ]:
from getpass import getpass

HUBSPOT_TOKEN = getpass("Paste your HubSpot access token: ")

In [ ]:
import requests

url = "https://api.hubapi.com/crm/v3/objects/contacts/search"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}",
    "Content-Type": "application/json"
}

all_contacts = []
after = None

while True:

    payload = {
        "limit": 100,
        "properties": [
            "firstname",
            "lastname",
            "email",
            "phone",
            "jobtitle",
            "company",
            "lifecyclestage"
        ],
        "filterGroups": [
            {
                "filters": [
                    {
                        "propertyName": "lifecyclestage",
                        "operator": "EQ",
                        "value": "lead"
                    }
                ]
            },
            {
                "filters": [
                    {
                        "propertyName": "lifecyclestage",
                        "operator": "EQ",
                        "value": "marketingqualifiedlead"
                    }
                ]
            },
            {
                "filters": [
                    {
                        "propertyName": "lifecyclestage",
                        "operator": "EQ",
                        "value": "salesqualifiedlead"
                    }
                ]
            }
        ]
    }

    if after is not None:
        payload["after"] = after

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    response = requests.post(
    url,
    headers=headers,
    json=payload
)

print(response.status_code)
print(response.text)


response.raise_for_status()

page = response.json()

all_contacts.extend(page.get("results", []))

# Look for another page
paging = page.get("paging")

if paging and "next" in paging:
    after = paging["next"]["after"]
else:
    break

print("Number of contacts returned:", len(all_contacts))

In [ ]:
# reference: https://developers.hubspot.com/docs/api-reference/latest/crm/search-the-crm#limits

url = "https://api.hubapi.com/crm/v3/objects/contacts/search"

headers = {
"Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

all_contacts = []
after = None

while True: 

    params = {
        "limit": 100,
        "properties": ",".join([
        "firstname",
        "lastname",
        "email",
        "phone",
        "jobtitle",
        "company",
        "lifecyclestage"
        ])

            }

if after is not None:
    params["after"] = after

response = requests.get(
url,
headers=headers,
params=params
)

response.raise_for_status()

page = response.json()
all_contacts.extend(page.get("results",[]))

print("Number of contacts returned:", len(hubspot_contacts["results"]))

In [ ]:
hs_contacts = []

for contact in hubspot_contacts["results"]:
    properties = contact["properties"]

    hs_contacts.append({
        "contact_id": contact["id"],
        "first_name": properties.get("firstname"),
        "last_name": properties.get("lastname"),
        "email": properties.get("email"),
        "phone": properties.get("phone"),
        "job_title": properties.get("jobtitle"),
        "company_name": properties.get("company"),
        "lifecycle_stage": properties.get("lifecyclestage"),
        "created_at": contact.get("createdAt"),
        "updated_at": contact.get("updatedAt")
    })

hubspot_contacts_df = pd.DataFrame(hs_contacts)

hubspot_contacts_df

#look at UN api and see if you can filter this for parameters

In [ ]:
url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

property_names = [
    "firstname",
    "lastname",
    "email",
    "phone",
    "jobtitle",
    "company",
    "lifecyclestage",
    "lead_source_1__c"
]

all_contacts = []
after = None

while True:
    params = {
        "limit": 100,
        "properties": ",".join(property_names)
    }

    if after is not None:
        params["after"] = after

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    page = response.json()
    all_contacts.extend(page.get("results", []))

    next_page = page.get("paging", {}).get("next")

    if not next_page:
        break

    after = next_page["after"]

print(f"Downloaded {len(all_contacts):,} contacts.")

In [ ]:
contact_rows = []

for contact in all_contacts:
    properties = contact.get("properties", {})

    contact_rows.append({
        "contact_id": contact.get("id"),
        "first_name": properties.get("firstname"),
        "last_name": properties.get("lastname"),
        "email": properties.get("email"),
        "phone": properties.get("phone"),
        "job_title": properties.get("jobtitle"),
        "company_name": properties.get("company"),
        "lifecycle_stage": properties.get("lifecyclestage"),
        "created_at": contact.get("createdAt"),
        "updated_at": contact.get("updatedAt"),
        "lead_source_1__c": properties.get("source")
    })

hubspot_contacts_df = pd.DataFrame(contact_rows)

hubspot_contacts_df